# Lesson 23 Lab — Kubernetes GPU Scheduling and Rollouts

**Puzzle:** Can a Deployment be highly available when every replica needs a scarce GPU and 30 GB of model state?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Kubernetes can restart processes and place pods, but it cannot create GPU capacity, shorten model loading, or make a single replica redundant. Requests, topology, probes, disruption budgets, and rollout surge must be designed around the inference lifecycle.


## 0. Predict before running

1. Calculate GPUs needed during a two-replica maxSurge rollout.
2. Check probe roles and grace period.
3. Choose the rollback signal before applying the manifest.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab renders and parses a minimal Deployment/Service/PDB configuration, then checks GPU requests/limits, rolling-update feasibility, readiness/startup probes, termination grace, cache strategy, and anti-affinity. No cluster is claimed.

- GPU resource requests are placement contracts.
- Liveness must not kill a healthy model during slow startup.
- A zero-downtime surge needs an actually free GPU.


## 2. Derive the mechanism

Device plugins advertise GPU resources and the scheduler treats them as indivisible. Readiness should wait for a loaded model; startup probes protect slow initialization; preStop and grace periods drain traffic. `maxSurge` requires spare GPU capacity, while `maxUnavailable` trades availability for an in-place rollout.

### Mechanism at a glance

```mermaid
flowchart TD
  D["Deployment revision"] --> S["scheduler: GPU + topology"]
  S --> P["pod starts and loads model"]
  P --> R{"readiness passes?"}
  R -->|"yes"| T["receive traffic"]
  R -->|"no"| W["stay out of Service"]
  T --> G["drain on termination"]
  G --> O["old pod removed"]
```

### Walk it step by step

1. **Request the device.** Declare one GPU resource per serving pod.
2. **Protect initialization.** Use startup and readiness probes with realistic model-load windows.
3. **Budget the rollout.** Ensure surge capacity exists or accept controlled unavailability.
4. **Drain and verify.** Stop new traffic, finish requests, and retain rollback evidence.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 23
LESSON_TITLE = 'Kubernetes GPU Scheduling and Rollouts'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260835
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | generic CPU-style Deployment defaults |
| Candidate | GPU-aware resources, probes, drain, topology, and rollout budget |
| Held constant | replica count, one GPU/pod, model load time, and declared cluster capacity |
| Measurements | manifest checks, steady GPUs, surge GPUs, capacity feasibility, probe presence, and native-cluster status |
| Evidence | `compatibility-probe` |

**Experiment:** Validate a GPU Deployment, Service, and disruption policy against rollout and lifecycle invariants.


## 5. Inspect the experiment code

The YAML is embedded and parsed into ordinary dictionaries. Each warning names the missing operational consequence rather than merely failing schema syntax.

Do not execute until the code matches the frozen table.


In [2]:
doc={"apiVersion":"apps/v1","kind":"Deployment","metadata":{"name":"vllm-qwen"},"spec":{
 "replicas":2,"strategy":{"type":"RollingUpdate","rollingUpdate":{"maxSurge":1,"maxUnavailable":0}},
 "template":{"metadata":{"labels":{"app":"vllm-qwen"}},"spec":{"terminationGracePeriodSeconds":120,
 "affinity":{"podAntiAffinity":{"preferredDuringSchedulingIgnoredDuringExecution":[{"weight":100}]}},
 "containers":[{"name":"server","image":"vllm/vllm-openai@sha256:"+"a"*64,
 "resources":{"requests":{"nvidia.com/gpu":1},"limits":{"nvidia.com/gpu":1}},
 "startupProbe":{"httpGet":{"path":"/health","port":8000},"failureThreshold":60,"periodSeconds":5},
 "readinessProbe":{"httpGet":{"path":"/health","port":8000},"periodSeconds":5},
 "livenessProbe":{"httpGet":{"path":"/health","port":8000},"periodSeconds":15}}]}}}}
spec=doc["spec"]; pod=spec["template"]["spec"]; container=pod["containers"][0]
requested=int(container["resources"]["requests"]["nvidia.com/gpu"]); limited=int(container["resources"]["limits"]["nvidia.com/gpu"])
replicas=int(spec["replicas"]); surge=int(spec["strategy"]["rollingUpdate"]["maxSurge"]); cluster_gpus=3
checks={"gpu_request_limit_match":requested==limited==1,"startup_probe":"startupProbe" in container,
 "readiness_probe":"readinessProbe" in container,"liveness_probe":"livenessProbe" in container,
 "termination_grace":pod.get("terminationGracePeriodSeconds",0)>=60,"anti_affinity":"affinity" in pod,
 "image_digest":"@sha256:" in container["image"],"rollout_capacity":replicas+surge<=cluster_gpus}
metrics={"checks":checks,"checks_passed":sum(checks.values()),"checks_total":len(checks),
 "capacity":{"steady_gpus":replicas*requested,"rollout_gpus":(replicas+surge)*requested,
             "cluster_gpus":cluster_gpus,"feasible":checks["rollout_capacity"]},
 "startup_probe":checks["startup_probe"],"manifest":doc,"native_cluster_executed":False}
analysis=(f"The manifest passed {metrics['checks_passed']}/{metrics['checks_total']} checks. Steady/"
          f"surge capacity is {metrics['capacity']['steady_gpus']}/{metrics['capacity']['rollout_gpus']} "
          f"of {cluster_gpus} declared GPUs. This is configuration feasibility, not a cluster rollout.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Checks passed | 8 |
| Checks total | 8 |
| Steady GPUs | 2 |
| Rollout GPUs | 3 |
| Capacity feasible | yes |
| Startup probe | yes |
| Native cluster executed | no |


## 7. Explain the result

The manifest passed 8/8 checks. Steady/surge capacity is 2/3 of 3 declared GPUs. This is configuration feasibility, not a cluster rollout.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The installed package/API/configuration surface was inspected. Availability or lint success is not equivalent to native feature execution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 23, "title": 'Kubernetes GPU Scheduling and Rollouts', "environment": ENV,
    "evidence_label": 'compatibility-probe', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The manifest audit establishes scheduling and rollout intent; Kubernetes availability remains unmeasured until a real cluster test.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 23,
  "title": "Kubernetes GPU Scheduling and Rollouts",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260835
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "checks": {
      "gpu_request_limit_match": true,
      "startup_probe": true,
      "readiness_probe": true,
      "liveness_probe": true,
      "termination_grace": true,
      "anti_affinity": true,
      "image_digest": true,
      "rollout_capacity": true
    },
    "checks_passed": 8,
    "checks_total": 8,
    "capacity": {
      "steady_gpus": 2,
      "rollout_gpus": 3,
      "cluster_gpus": 3,
      "feasible": true
    },
    "startup_probe": true,
    "manifest": {
      "apiVersion": "apps/v1",
      "kind": "Deployment",
      "metadata": {
        "name": "vllm-qwen"
      },
     

## 9. Make the bounded decision

> The manifest audit establishes scheduling and rollout intent; Kubernetes availability remains unmeasured until a real cluster test.

**Acceptance/rollback:** Apply only when steady and rollout GPU capacity, probes, drain, disruption, metrics, and rollback revision are all demonstrated in staging.

**Failure analysis:** Static configuration cannot verify device plugin health, image/model pull time, scheduler fragmentation, node failures, or actual probe behavior.


## 10. Extend the evidence

Deploy to a staging cluster, delete pods/nodes during load, run a canary, test rollout with saturated GPUs, and measure time to readiness and traffic drain.

The full boundary and references are in [`README.md`](README.md).
